(mesh-nb)=
# `Mesh` class

This tutorial will show you how to create custom meshes and winding paths around the the combined Brillouin zone (BZ) and parameter space.

In [1]:
from pythtb import Mesh
import numpy as np

`Mesh` accepts the following parameters:
- `dim_k`: Integer number of k-space dimensions (1, 2, or 3)
- `axis_types`: List of strings defining the type of each axis; options are `"k"` for k-space axes and `"l"` for parameter space ($\lambda$) axes
- `dim_lambda`: (optional): Integer number of parameter space dimensions. If not provided, it is inferred from the number of `"l"` axes in `axis_types`.
- `axis_names`: (optional): List of strings defining the names of each axis, primarily used for adiabatic $\lambda$ cycles (see Three-site Thouless pump for example)

In the following cell, we create a mesh that on a 2D k-space with a single k-axis. This means that this is a path in the 2D BZ. We will use the method
`build_custom()` to specifiy the mesh points directly. Here we create a path from the $\Gamma$ = `[0, 0]` point to the $X = $ `[0.5, 0.5]` point in the square BZ.

In [2]:
mesh = Mesh(dim_k=2, axis_types=['k'])
points = np.linspace([0,0], [1, 1], 10, endpoint=False)  # path from (0,0) to (1, 1)
mesh.build_custom(points)

We access the mesh points via the `points` attribute. Notice that each point is a 2D vector corresponding to the two k-space dimensions.

In [3]:
print(mesh.points)

[[0.  0. ]
 [0.1 0.1]
 [0.2 0.2]
 [0.3 0.3]
 [0.4 0.4]
 [0.5 0.5]
 [0.6 0.6]
 [0.7 0.7]
 [0.8 0.8]
 [0.9 0.9]]


We can see the mesh information by printing the `Mesh` object. This tells us 

- the type of mesh (either "path" or "grid")
- the number of dimensions in k-space and parameter space
- The total number of mesh points
- The shape of the mesh array
- The k-axes (type (k), name, and number of points)
- The parameter space axes (type (l), name, and number of points)
- The axes that are looped (periodic) or not
- The axes that wind around the BZ/parameter space
- The axes that include the endpoints (are closed) or not. This is only relevant for winding or looped paths.

The last three properties are important for determining how the mesh wraps around the BZ and parameter space when calculating topological invariants.

In [4]:
print(mesh)

Mesh Summary
Type: path
Dimensionality: 2 k-dim(s) + 0 λ-dim(s)
Number of mesh points: 10
Full shape: (10, 2)
k-axes: [Axis(type=k, name=k_0, size=10)]
λ-axes: []
Looped axes: None
BZ-winding axes: None
Endpoint axes: None


The code doesn't inherently know which axes should wind the BZ or parameter space if the endpoints are not included, so we have to specify this manually. This is important for poperly handling the periodic boundary conditions with the Bloch wavefunctions. 

We can declare an axis as winding the BZ by calling the method `wind_bz`. We pass two arguments:

- `axis_idx`: The index of the axis to wind (0-indexed)
- `component_idx`: The index of the component of the combined ($k$, $\lambda$) vector

We set `axis_idx=0` to wind the first (and only) axis of our mesh, and `component_idx=0` to wind around the first k-space dimension. After calling this method, we can see that the first axis is now marked as winding the BZ. We do the same thing for the second k-space dimension by calling `wind_bz` again with `component_idx=1`. 

Note that a BZ winding automatically implies that the axis is looped (periodic). The reason for this distinction is that there may be cases where an axis is looped but does not wind the BZ, such as in parameter space cycles, or loops in k-space that do not traverse the full BZ.


In [5]:
mesh.wind_bz(axis_idx=0, component_idx=0)  # mark as winding k_x
mesh.wind_bz(axis_idx=0, component_idx=1)  # mark as winding k_y
print(mesh)

Mesh Summary
Type: path
Dimensionality: 2 k-dim(s) + 0 λ-dim(s)
Number of mesh points: 10
Full shape: (10, 2)
k-axes: [Axis(type=k, name=k_0, size=10)]
λ-axes: []
Looped axes: (axis 0 loops component 0), (axis 0 loops component 1)
BZ-winding axes: (axis 0 winds component 0), (axis 0 winds component 1)
Endpoint axes: None


The class will automatically detect an axis as winding the BZ if the start and end points are the same modulo 1 (i.e. the axis is closed) for the k-components. This is useful for paths that explicitly include the endpoints, such as circular paths in k-space or closed loops in parameter space.

We will do the same thing as we have done above, but this time include the endpoints (1). This means that the start and end points are the same, so the class will automatically detect that both axes wind the BZ without needing to call `wind_bz` manually.

Notice that the Mesh also detects that the axis contains the endpoints (is closed).

In [6]:
mesh = Mesh(dim_k=2, axis_types=['k'])
points = np.linspace([0,0], [1, 1], 10, endpoint=True)  # path from (0,0) to (1, 1)
mesh.build_custom(points)
print(mesh)

Mesh Summary
Type: path
Dimensionality: 2 k-dim(s) + 0 λ-dim(s)
Number of mesh points: 10
Full shape: (10, 2)
k-axes: [Axis(type=k, name=k_0, size=10)]
λ-axes: []
Looped axes: (axis 0 loops component 0), (axis 0 loops component 1)
BZ-winding axes: (axis 0 winds component 0), (axis 0 winds component 1)
Endpoint axes: (axis 0 contains endpoint of component 0), (axis 0 contains endpoint of component 1)
